In [5]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root / "src"))

import os
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.distributions.Distribution.set_default_validate_args(False)

from sbi.analysis import pairplot

import sbi_for_diffusion_models
from sbi_for_diffusion_models.priors import build_prior_theta
from sbi_for_diffusion_models.mnpe import train_npe_session, run_inference_npe, run_sbc_npe
from sbi_for_diffusion_models.data_simulator import simulate_training_sessions
from sbi_for_diffusion_models.run_config import RUN_CONFIG_PARAMS
from sbi_for_diffusion_models.models.rt_choice_model import (
    simulate_rt_choice_batch,
    pack_x_rt_choice,
    generate_pulses_torch,
    max_num_pulses
)

In [6]:
cfg = RUN_CONFIG_PARAMS
device = "cpu"
dev = torch.device(device)

P = max_num_pulses()
T = int(cfg.NUM_TRIALS_OBS)
trial_dim = 2 + P
print(f"P={P}, T={T}, trial_dim={trial_dim}")

prior_theta = build_prior_theta()
if hasattr(prior_theta, "to"):
    prior_theta.to(dev)

P=34, T=256, trial_dim=36


In [7]:
# test simulation 
from sbi_for_diffusion_models.mnpe import  _build_npe_embedding_net, _build_npe_estimator_builder

P = max_num_pulses()
T = int(cfg.NUM_TRIALS_OBS)

embedding_net = _build_npe_embedding_net(cfg, T=T, P=P).to(dev)
est_builder = _build_npe_estimator_builder(cfg, embedding_net)

num_steps = int(getattr(cfg, "NPE_NUM_STEPS", 10_000))

In [8]:
density_estimator, posterior_obj = train_npe_session(
    cfg, prior_theta, device=device, seed=0
)
print("density_estimator device:", next(density_estimator.parameters()).device)

[NPE] device=cpu, sess_per_step=512, num_steps=500, lr=0.0005
step 50: loss=90656.3984 ema=104472.4006
step 100: loss=56017.9648 ema=84000.0958


RuntimeError: [prior predictive check failed] Too many timeout trials after retries.
Session 229: timeouts=66/256 (25.8%), allowed=52/256 (20%).
MAX_TIMEOUT_TRIES=20
This likely indicates a bad prior (e.g., drift too small, bounds too large, tau too close to T_MAX, etc.). Choose a better prior.